# 面试题：为什么 Transformer 必须有位置编码，如何手写并验证它真的区分顺序？

## 面试回答主线

纯 self-attention 对非特殊 token 的排列是等变的：交换输入顺序只会交换对应输出，固定 `[CLS]` 对同一 token 多重集得到的汇聚结果不变。位置编码给每个位置注入不同信号，使注意力能够学习“谁在谁前面”。正弦位置编码不需要训练，偶数维用 `sin`、奇数维用 `cos`，不同频率让模型既感知局部距离也表达长位置。绝对位置并不自动处理 padding，尤其左 padding 时必须根据 attention mask 生成 position id。长上下文、外推和 KV cache 还要求位置规则在训练与增量解码之间完全一致。

## 真实案例：合同关系方向分类

任务判断“甲方是否为动作发起者”。每对文本拥有完全相同的 token 多重集，仅交换甲乙位置；这是关系抽取中最小但真实的方向性问题。数据为教学构造，不代表法律文本模型效果。

In [1]:
import math  # 导入对数和指数常数以构造正弦频率。
import torch  # 导入 PyTorch 以实现位置编码和真实注意力训练。
from torch import nn  # 导入基础模块与参数抽象。
import torch.nn.functional as F  # 导入稳定的二分类损失函数。
torch.set_num_threads(1)  # 小张量教学实验固定单线程以避免多线程调度开销。
examples = [  # 构造只靠顺序才能区分标签的合同关系样本。
    (["[CLS]", "甲方", "收购", "乙方"], 1),  # 甲方位于谓词前并发起收购。
    (["[CLS]", "乙方", "收购", "甲方"], 0),  # 相同 token 逆序后甲方变为客体。
    (["[CLS]", "甲方", "起诉", "乙方"], 1),  # 甲方是起诉动作的发起者。
    (["[CLS]", "乙方", "起诉", "甲方"], 0),  # 交换实体后甲方是被起诉方。
    (["[CLS]", "甲方", "投资", "乙方"], 1),  # 甲方是投资方。
    (["[CLS]", "乙方", "投资", "甲方"], 0),  # 甲方在逆序句中是被投资方。
    (["[CLS]", "甲方", "委托", "乙方"], 1),  # 甲方是委托动作发起者。
    (["[CLS]", "乙方", "委托", "甲方"], 0),  # 甲方在逆序句中是受托方。
    (["[CLS]", "甲方", "控股", "乙方"], 1),  # 甲方是控股主体。
    (["[CLS]", "乙方", "控股", "甲方"], 0),  # 甲方在逆序句中是被控股主体。
]  # 结束方向分类样本列表。
print("文本                     标签  含义")  # 输出真实案例输入表标题。
for tokens, label in examples:  # 逐条展示原始 token 顺序和监督信号。
    meaning = "甲方是发起者" if label == 1 else "甲方是承受者"  # 把二元标签转换为可读业务含义。
    print(f"{' '.join(tokens):<24} {label:>4}  {meaning}")  # 输出文本、标签和方向解释。

文本                     标签  含义
[CLS] 甲方 收购 乙方              1  甲方是发起者
[CLS] 乙方 收购 甲方              0  甲方是承受者
[CLS] 甲方 起诉 乙方              1  甲方是发起者
[CLS] 乙方 起诉 甲方              0  甲方是承受者
[CLS] 甲方 投资 乙方              1  甲方是发起者
[CLS] 乙方 投资 甲方              0  甲方是承受者
[CLS] 甲方 委托 乙方              1  甲方是发起者
[CLS] 乙方 委托 甲方              0  甲方是承受者
[CLS] 甲方 控股 乙方              1  甲方是发起者
[CLS] 乙方 控股 甲方              0  甲方是承受者


## Baseline（基线）：忽略顺序的 bag key 必然碰撞

把非 `[CLS]` token 排序后作为特征，每一对正反句都会得到相同 key。相同特征对应一正一负，任何确定性分类器最多只能猜中一半。

In [2]:
groups = {}  # 创建无序 token key 到标签列表的映射。
for tokens, label in examples:  # 遍历全部方向分类样本。
    bag_key = tuple(sorted(tokens[1:]))  # 丢弃位置并排序得到 bag-of-tokens 特征。
    groups.setdefault(bag_key, []).append(label)  # 聚合同一无序特征对应的冲突标签。
baseline_predictions = []  # 保存每条样本的无序基线预测。
for tokens, label in examples:  # 再次遍历样本以按所属组进行多数表决。
    bag_key = tuple(sorted(tokens[1:]))  # 重新计算与训练阶段一致的无序 key。
    prediction = int(sum(groups[bag_key]) >= len(groups[bag_key]) / 2)  # 并列时固定预测一以保证确定性。
    baseline_predictions.append(prediction)  # 保存当前样本的基线预测。
baseline_accuracy = sum(prediction == label for prediction, (_, label) in zip(baseline_predictions, examples)) / len(examples)  # 计算无序方案的样本准确率。
print("无序特征                              冲突标签")  # 输出 bag key 冲突表标题。
for bag_key, labels in sorted(groups.items()):  # 逐组展示相同特征如何对应相反方向。
    print(f"{bag_key!s:<38} {labels}")  # 输出无序 token 集合和真实标签列表。
print(f"忽略顺序的训练集准确率：{baseline_accuracy:.1%}")  # 输出理论上限在本数据上的实际结果。

无序特征                              冲突标签
('乙方', '委托', '甲方')                     [1, 0]
('乙方', '投资', '甲方')                     [1, 0]
('乙方', '控股', '甲方')                     [1, 0]
('乙方', '收购', '甲方')                     [1, 0]
('乙方', '甲方', '起诉')                     [1, 0]
忽略顺序的训练集准确率：50.0%


## 核心实现一：从公式手写 sinusoidal position encoding

第 `pos` 行、第 `2i` 维为 `sin(pos / 10000^(2i/d))`，第 `2i+1` 维为对应 `cos`。位置零的偶数维为 0、奇数维为 1；频率随维度降低。下面打印真实矩阵和相邻位置余弦。

In [3]:
def sinusoidal_encoding(length, dimension):  # 按原始 Transformer 公式生成固定位置编码矩阵。
    positions = torch.arange(length, dtype=torch.float32).unsqueeze(1)  # 创建从零开始的列向量位置索引。
    even_dimensions = torch.arange(0, dimension, 2, dtype=torch.float32)  # 取得所有偶数维索引以共享频率。
    inverse_frequencies = torch.exp(-math.log(10000.0) * even_dimensions / dimension)  # 计算从高频到低频的几何频率。
    angles = positions * inverse_frequencies.unsqueeze(0)  # 计算每个位置与每个频率对应的相位。
    encoding = torch.zeros(length, dimension)  # 创建最终位置编码矩阵。
    encoding[:, 0::2] = torch.sin(angles)  # 把正弦值写入全部偶数维。
    encoding[:, 1::2] = torch.cos(angles[:, : encoding[:, 1::2].shape[1]])  # 把余弦值写入全部奇数维并兼容奇数维度。
    return encoding  # 返回可直接加到 token embedding 的固定矩阵。
position_preview = sinusoidal_encoding(6, 8)  # 为六个位置生成八维教学矩阵。
print("位置编码矩阵（行=position，列=dimension）：")  # 输出矩阵含义说明。
for index, row in enumerate(position_preview):  # 逐位置展示便于观察频率变化。
    print(f"pos={index}: " + " ".join(f"{float(value):>7.3f}" for value in row))  # 输出保留三位小数的位置向量。
normalized_positions = F.normalize(position_preview, dim=1)  # 对每个位置向量归一化以比较方向相似度。
print("pos0 与各位置余弦：", [round(float(value), 3) for value in normalized_positions @ normalized_positions[0]])  # 输出随距离变化的真实相似度。

位置编码矩阵（行=position，列=dimension）：
pos=0:   0.000   1.000   0.000   1.000   0.000   1.000   0.000   1.000
pos=1:   0.841   0.540   0.100   0.995   0.010   1.000   0.001   1.000
pos=2:   0.909  -0.416   0.199   0.980   0.020   1.000   0.002   1.000
pos=3:   0.141  -0.990   0.296   0.955   0.030   1.000   0.003   1.000
pos=4:  -0.757  -0.654   0.389   0.921   0.040   0.999   0.004   1.000
pos=5:  -0.959   0.284   0.479   0.878   0.050   0.999   0.005   1.000
pos0 与各位置余弦： [1.0, 0.884, 0.641, 0.491, 0.567, 0.79]


## 核心实现二：手写单头 `[CLS]` attention 并真实训练

模型只用 `nn.Parameter` 定义 token 表、Q/K/V 投影和分类头，`forward` 明确计算注意力分数与加权和。我们从相同随机种子分别训练“无位置”和“正弦位置”版本；若实现正确，无位置模型对每组逆序句输出完全相同。

In [4]:
tokens = sorted({token for sequence, _ in examples for token in sequence})  # 收集样本中的完整 token 集合。
token_to_id = {token: index for index, token in enumerate(tokens)}  # 创建稳定 token id 映射。
input_ids = torch.tensor([[token_to_id[token] for token in sequence] for sequence, _ in examples], dtype=torch.long)  # 把全部样本转换为批量整数张量。
labels = torch.tensor([label for _, label in examples], dtype=torch.float32)  # 把方向标签转换为浮点监督张量。
class TinyOrderAttention(nn.Module):  # 定义用于验证位置必要性的最小单头注意力分类器。
    def __init__(self, vocabulary_size, dimension, use_position):  # 初始化 token 表、投影矩阵和位置开关。
        super().__init__()  # 注册基础模块状态以追踪所有参数。
        self.dimension = dimension  # 保存隐藏维度供 attention 缩放和位置编码使用。
        self.use_position = use_position  # 保存当前实验是否注入位置编码。
        self.token_weight = nn.Parameter(torch.randn(vocabulary_size, dimension) * 0.15)  # 创建可学习 token embedding 表。
        self.query_weight = nn.Parameter(torch.randn(dimension, dimension) * 0.15)  # 创建查询线性投影矩阵。
        self.key_weight = nn.Parameter(torch.randn(dimension, dimension) * 0.15)  # 创建键线性投影矩阵。
        self.value_weight = nn.Parameter(torch.randn(dimension, dimension) * 0.15)  # 创建值线性投影矩阵。
        self.classifier_weight = nn.Parameter(torch.randn(dimension) * 0.15)  # 创建二分类输出权重。
        self.classifier_bias = nn.Parameter(torch.zeros(()))  # 创建标量分类偏置。
    def forward(self, ids):  # 执行 embedding、单头 self-attention 和二分类。
        hidden = self.token_weight[ids]  # 按 token id 查表得到批量序列向量。
        if self.use_position:  # 仅在位置实验中注入固定正弦信号。
            hidden = hidden + sinusoidal_encoding(ids.shape[1], self.dimension).unsqueeze(0)  # 给每个 batch 样本加相同位置矩阵。
        cls_hidden = hidden[:, :1, :]  # 取固定首位 [CLS] 作为唯一查询位置。
        queries = cls_hidden @ self.query_weight  # 计算 [CLS] 查询向量。
        keys = hidden @ self.key_weight  # 计算序列每个位置的键向量。
        values = hidden @ self.value_weight  # 计算序列每个位置的值向量。
        scores = (queries * keys).sum(dim=2) / math.sqrt(self.dimension)  # 计算缩放点积注意力分数。
        attention = torch.softmax(scores, dim=1)  # 沿序列维度归一化得到注意力权重。
        context = (attention.unsqueeze(2) * values).sum(dim=1)  # 对值向量执行注意力加权和。
        logits = context @ self.classifier_weight + self.classifier_bias  # 把 [CLS] 上下文映射为二分类 logit。
        return logits, attention  # 返回预测分数和可解释的注意力权重。
def train_order_model(use_position):  # 用手动 SGD 训练一个位置设置固定的模型。
    torch.manual_seed(41)  # 对两个实验使用完全相同的参数初始化。
    current_model = TinyOrderAttention(len(tokens), 12, use_position)  # 创建十二维单头注意力模型。
    trace = []  # 保存代表轮次的损失和准确率。
    for step in range(1601):  # 执行足够轮次让小型方向任务形成清晰分类间隔。
        logits, attention = current_model(input_ids)  # 调用自定义 forward 计算全批量预测。
        loss = F.binary_cross_entropy_with_logits(logits, labels)  # 计算稳定的二分类交叉熵。
        loss.backward()  # 对 token 表、Q/K/V 和分类头执行真实反向传播。
        with torch.no_grad():  # 关闭参数更新过程的梯度记录。
            for parameter in current_model.parameters():  # 逐个遍历本模型的所有可学习参数。
                parameter -= 0.08 * parameter.grad  # 使用固定学习率执行手动 SGD。
                parameter.grad.zero_()  # 清空本轮梯度避免错误累积。
        if step in {0, 20, 100, 300, 800, 1600}:  # 只保留能够反映收敛趋势与最终间隔的关键轮次。
            predictions = (torch.sigmoid(logits.detach()) >= 0.5).to(torch.float32)  # 把当前概率阈值化为类别预测。
            accuracy = float((predictions == labels).to(torch.float32).mean())  # 计算当前全批量准确率。
            trace.append((step, float(loss.detach()), accuracy))  # 保存轮次、损失和准确率。
    return current_model, trace  # 返回训练模型及其可解释轨迹。
model_without_position, trace_without_position = train_order_model(False)  # 训练没有任何位置信号的对照模型。
model_with_position, trace_with_position = train_order_model(True)  # 从同初始化训练加入正弦位置的模型。
print("轮次 | 无位置loss/acc | 有位置loss/acc")  # 输出两种设置的训练轨迹表标题。
for no_position_row, position_row in zip(trace_without_position, trace_with_position):  # 对齐相同轮次的实验结果。
    print(f"{no_position_row[0]:>4} | {no_position_row[1]:>7.4f}/{no_position_row[2]:>5.1%} | {position_row[1]:>7.4f}/{position_row[2]:>5.1%}")  # 输出损失与准确率对照。

轮次 | 无位置loss/acc | 有位置loss/acc
   0 |  0.6932/50.0% |  0.6978/50.0%
  20 |  0.6932/50.0% |  0.6935/50.0%
 100 |  0.6932/50.0% |  0.6931/50.0%
 300 |  0.6932/50.0% |  0.6928/50.0%
 800 |  0.6932/50.0% |  0.6768/100.0%
1600 |  0.6932/50.0% |  0.0021/100.0%


## 结果表：逆序句的 logit 与注意力

无位置模型不仅“效果差”，而是每对逆序句的 logit 在数值上相同；位置模型能够把实体所在位置纳入判断。下面逐样本打印概率和位置模型的注意力分布。

In [5]:
with torch.no_grad():  # 评估阶段关闭梯度图以得到稳定输出。
    logits_without, attention_without = model_without_position(input_ids)  # 计算无位置模型的全量预测。
    logits_with, attention_with = model_with_position(input_ids)  # 计算位置模型的全量预测和注意力。
    probabilities_without = torch.sigmoid(logits_without)  # 把无位置 logit 转换为正类概率。
    probabilities_with = torch.sigmoid(logits_with)  # 把有位置 logit 转换为正类概率。
    predictions_without = (probabilities_without >= 0.5).to(torch.float32)  # 对无位置概率执行固定阈值决策。
    predictions_with = (probabilities_with >= 0.5).to(torch.float32)  # 对有位置概率执行固定阈值决策。
accuracy_without = float((predictions_without == labels).to(torch.float32).mean())  # 计算无位置模型最终准确率。
accuracy_with = float((predictions_with == labels).to(torch.float32).mean())  # 计算位置模型最终准确率。
print("文本                     标签  无位置P  有位置P  有位置attention")  # 输出逐样本结果表标题。
for index, (sequence, label) in enumerate(examples):  # 遍历全部关系方向样本。
    attention_view = [round(float(value), 3) for value in attention_with[index]]  # 格式化当前位置模型的注意力权重。
    print(f"{' '.join(sequence):<24} {label:>4} {float(probabilities_without[index]):>8.3f} {float(probabilities_with[index]):>8.3f}  {attention_view}")  # 输出标签、两种概率与注意力。
pair_logit_gaps = [abs(float(logits_without[index] - logits_without[index + 1])) for index in range(0, len(examples), 2)]  # 计算无位置模型每对逆序句的 logit 差。
print(f"最终准确率：无位置={accuracy_without:.1%}，有位置={accuracy_with:.1%}")  # 输出核心效果对照。
print("无位置模型每对逆序句 logit 差：", [f"{gap:.8f}" for gap in pair_logit_gaps])  # 输出排列不变性的直接数值证据。

文本                     标签  无位置P  有位置P  有位置attention
[CLS] 甲方 收购 乙方              1    0.502    0.998  [0.003, 0.985, 0.012, 0.001]
[CLS] 乙方 收购 甲方              0    0.502    0.002  [0.007, 0.004, 0.031, 0.957]
[CLS] 甲方 起诉 乙方              1    0.498    0.998  [0.003, 0.992, 0.005, 0.001]
[CLS] 乙方 起诉 甲方              0    0.498    0.002  [0.007, 0.004, 0.013, 0.975]
[CLS] 甲方 投资 乙方              1    0.503    0.998  [0.003, 0.988, 0.009, 0.001]
[CLS] 乙方 投资 甲方              0    0.503    0.002  [0.007, 0.004, 0.023, 0.966]
[CLS] 甲方 委托 乙方              1    0.499    0.998  [0.003, 0.987, 0.01, 0.001]
[CLS] 乙方 委托 甲方              0    0.499    0.002  [0.007, 0.004, 0.026, 0.963]
[CLS] 甲方 控股 乙方              1    0.498    0.998  [0.003, 0.987, 0.01, 0.001]
[CLS] 乙方 控股 甲方              0    0.498    0.002  [0.007, 0.004, 0.025, 0.964]
最终准确率：无位置=50.0%，有位置=100.0%
无位置模型每对逆序句 logit 差： ['0.00000000', '0.00000000', '0.00000000', '0.00000000', '0.00000000']


## 结果解读

固定 `[CLS]` 查询面对相同 token 多重集时，无位置 attention 的 key/value 只是被重排，加权和不变，因此每对逆序句 logit 差接近浮点误差。正弦信号让“甲方在谓词前”和“甲方在谓词后”成为不同输入，模型才可能拟合方向标签。这个实验验证的是表示能力，不是声称十条样本足以训练合同分类器。

## 失败案例：左 padding 后仍直接使用 `arange`

若未补齐句子的内容位置是 `[0,1,2]`，左补两个 PAD 后直接用物理下标会变成 `[2,3,4]`，同一内容得到不同绝对位置向量。修复方法是由 attention mask 的累计和生成逻辑 position id，使第一个真实 token 仍为位置零。

In [6]:
unpadded_encoding = sinusoidal_encoding(3, 8)  # 生成未补齐三 token 句子的参考位置向量。
naive_padded_content = sinusoidal_encoding(5, 8)[2:]  # 错误地让左补齐后的真实内容占据物理位置二到四。
left_padding_mask = torch.tensor([0, 0, 1, 1, 1], dtype=torch.long)  # 用零标记 PAD、用一标记三个真实 token。
logical_position_ids = (left_padding_mask.cumsum(dim=0) - 1).clamp_min(0)  # 通过 mask 累计和恢复从零开始的内容逻辑位置。
position_table = sinusoidal_encoding(5, 8)  # 创建覆盖最大逻辑位置的正弦表。
fixed_padded_content = position_table[logical_position_ids][left_padding_mask.bool()]  # 仅选择真实 token 对应的逻辑位置向量。
naive_shift = float((unpadded_encoding - naive_padded_content).abs().max())  # 量化错误物理位置造成的最大表示漂移。
fixed_shift = float((unpadded_encoding - fixed_padded_content).abs().max())  # 量化 mask 位置 id 修复后的剩余差异。
print("未补齐 position ids：", [0, 1, 2])  # 输出参考句子的逻辑位置。
print("错误左补齐内容 ids：", [2, 3, 4], f"，最大向量漂移={naive_shift:.4f}")  # 展示直接 arange 的错误行为。
print("mask 生成的全部 ids：", logical_position_ids.tolist())  # 展示 PAD 与内容位置的实际编号。
print("修复后内容 ids：", logical_position_ids[left_padding_mask.bool()].tolist(), f"，最大向量漂移={fixed_shift:.4f}")  # 展示内容位置恢复一致后的结果。

未补齐 position ids： [0, 1, 2]
错误左补齐内容 ids： [2, 3, 4] ，最大向量漂移=1.6661
mask 生成的全部 ids： [0, 0, 0, 1, 2]
修复后内容 ids： [0, 1, 2] ，最大向量漂移=0.0000


## 生产差距与落地清单

教学模型是单头、短序列并在训练集上验证表示能力。线上需同时处理 attention mask、左/右 padding、截断、position offset、KV cache 增量位置和最大上下文；正弦、可学习绝对位置、RoPE、ALiBi 的外推行为不同，不能只换一个函数。发布前应做顺序最小对、不同 padding 长度一致性、全量与增量解码一致性、长于训练长度的压力测试，并监控真实长度分布。

## 最小回归测试

断言只固定公式、排列不变性、方向可学习性和 padding 位置合同；矩阵、训练曲线和逐样本概率才是主要证据。

In [7]:
assert torch.allclose(position_preview[0, 0::2], torch.zeros(4))  # 验证位置零的全部偶数维正弦值为零。
assert torch.allclose(position_preview[0, 1::2], torch.ones(4))  # 验证位置零的全部奇数维余弦值为一。
assert max(pair_logit_gaps) < 1e-6  # 验证无位置 [CLS] attention 对逆序非特殊 token 保持不变。
assert accuracy_without == 0.5  # 验证相同 bag 对应冲突标签时无位置模型只能达到一半准确率。
assert accuracy_with >= 0.9  # 验证加入位置编码后模型能够学习关系方向。
assert naive_shift > 0.5  # 固化左 padding 直接使用物理下标会显著漂移的失败案例。
assert fixed_shift < 1e-7  # 验证 mask 派生逻辑 position id 后内容表示完全一致。
print("最小回归测试通过：正弦公式、顺序区分和左 padding 位置修复均符合预期。")  # 输出完整顺序执行成功的明确结论。

最小回归测试通过：正弦公式、顺序区分和左 padding 位置修复均符合预期。
